# 🧠 Panopticon Protocol v3 — HuggingFace TRL Fine-Tuning

> **Team:** Ayush Kumar & Ravi Prashant | **Target:** Meta PyTorch OpenEnv Grand Finale
>
> This notebook executes Supervised Fine-Tuning (SFT) on a Large Language Model using HuggingFace TRL to master the Panopticon Protocol counter-espionage environment. We use LoRA to efficiently fine-tune a model to perform long-horizon reasoning, identify false flags, and eliminate sleeper agents.

### Judging Criteria Fulfilled:
- ✅ **Theme #1 & #2 Alignment**: Trains theory-of-mind and multi-step reasoning.
- ✅ **HF TRL Usage**: Uses `SFTTrainer` with LoRA adapters.
- ✅ **Real Training**: Generates expert trajectories and fine-tunes the LLM.

⚠️ **REQUIREMENT**: Ensure you are connected to a GPU Runtime (`Runtime → Change runtime type → T4 GPU` or better).

## 1️⃣ Setup & Authentication

First, we will install the necessary dependencies, clone the OpenEnv environment, and authenticate with HuggingFace using your pre-configured $30 compute credits token.

In [ ]:
# Install HuggingFace TRL and OpenEnv dependencies
!pip install -q torch trl transformers peft accelerate datasets gymnasium fastapi pydantic httpx tqdm numpy

In [ ]:
import os
from huggingface_hub import login

# Pre-configured with your OpenEnv Hackathon Token
HF_TOKEN = "<PASTE_YOUR_HF_TOKEN_HERE>"
USERNAME = "Ayush-Kumar0207"
os.environ['HF_TOKEN'] = HF_TOKEN

# Log in to HuggingFace
login(token=HF_TOKEN, add_to_git_credential=True)
print(f"✅ Successfully logged into Hugging Face as {USERNAME}")

In [ ]:
# Clone the Panopticon Protocol environment repository
!rm -rf /content/panopticon-protocol-v3
!git clone https://github.com/Ayush-Kumar0207/OpenEnv-Starter-Kit.git panopticon-protocol-v3
%cd /content/panopticon-protocol-v3
!ls -la *.py

## 2️⃣ Generate Expert Trajectories

To train the LLM, we need examples of "perfect play." The `train_trl.py` script contains a heuristic agent that plays the game optimally. We will run it to generate JSONL files mapping `Environment State (Text)` → `Correct Action (JSON)`.

In [ ]:
# We will target the Qwen 2.5 1.5B Instruct model since we have compute credits.
# Let's first generate trajectories for the Level 5 (Manchurian) difficulty.

# Generate 50 expert episodes for the hardest difficulty to build the dataset.
!python train_trl.py --level level_5 --episodes 50 --model Qwen/Qwen2.5-1.5B-Instruct

## 3️⃣ Curriculum Supervised Fine-Tuning (SFT)

Now we perform the actual fine-tuning. We use **LoRA** (Low-Rank Adaptation) to efficiently train the LLM without updating all 1.5B parameters. 

By passing the `--curriculum` flag, the script will:
1. Generate data for `easy` mode and train the model.
2. Generate data for `medium` mode and train the model further.
3. Progress all the way to `level_5` (Manchurian).

*Note: This will take significant GPU time. If you want a quick test, run without `--curriculum`.*

In [ ]:
# Execute Curriculum Training on Qwen 2.5 1.5B
# WARNING: This can take 1-2 hours depending on the GPU.
!python train_trl.py --curriculum --model Qwen/Qwen2.5-1.5B-Instruct --episodes 20

## 4️⃣ Save & Push to HuggingFace Hub

Once training is complete, the LoRA adapters are saved locally. 
This script pushes the final Level 5 trained model directly to your HuggingFace account so it can be loaded instantly by your Gradio Space or inference endpoints.

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()
repo_id = f"{USERNAME}/panopticon-argus-qwen-1.5B"

try:
    # Create the repository if it doesn't exist
    api.create_repo(repo_id, exist_ok=True)
    print(f"Created/Verified repository: {repo_id}")
    
    # Push the level_5 model (the final product of the curriculum)
    if os.path.exists("trl_model_level_5"):
        print("Pushing model to Hugging Face Hub... this may take a moment.")
        api.upload_folder(
            folder_path="trl_model_level_5",
            repo_id=repo_id,
            commit_message="Upload fully trained Panopticon Protocol agent (Level 5)"
        )
        print(f"✅ Successfully pushed model to https://huggingface.co/{repo_id}")
    else:
        print("⚠️ Model folder 'trl_model_level_5' not found. Ensure training completed successfully.")
except Exception as e:
    print(f"❌ Failed to push to Hub: {e}")